In [4]:
# 📦 Cell 1 — Upload ZIP, load CSV, show 5 head & 5 tail

import zipfile
import pandas as pd
from google.colab import files
import os

print("🔼 لطفاً فایل ZIP را آپلود کن...")
uploaded = files.upload()

if not uploaded:
    raise SystemExit("⛔ هیچ فایلی آپلود نشد.")

zip_filename = list(uploaded.keys())[0]
print(f"✅ ZIP دریافت شد: {zip_filename}")

# 1) Extract ZIP
extract_dir = "csv_file"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall(extract_dir)

# 2) Find the CSV inside ZIP
csv_path = None
for root, dirs, files_list in os.walk(extract_dir):
    for f in files_list:
        if f.endswith(".csv"):
            csv_path = os.path.join(root, f)
            break

if csv_path is None:
    raise SystemExit("⛔ هیچ CSV داخل ZIP پیدا نشد!")

print(f"📄 CSV پیدا شد: {csv_path}")

# 3) Load CSV
df = pd.read_csv(csv_path, engine="python", on_bad_lines="skip")
print(f"✅ CSV بارگذاری شد. تعداد ردیف‌ها: {len(df)}, ستون‌ها: {df.shape[1]}")

# 4) Preview
print("\n👀 پنج ردیف اول:")
display(df.head(5))

print("\n👀 پنج ردیف آخر:")
display(df.tail(5))


🔼 لطفاً فایل ZIP را آپلود کن...


Saving merged (9).zip to merged (9).zip
✅ ZIP دریافت شد: merged (9).zip
📄 CSV پیدا شد: csv_file/merged (9).csv
✅ CSV بارگذاری شد. تعداد ردیف‌ها: 3742439, ستون‌ها: 10

👀 پنج ردیف اول:


,@timestamp,user.name,host.hostname,source.ip,event.category,event.action,winlog.logon.type,winlog.logon.id,winlog.event_data.LogonProcessName,winlog.task
0,"Nov 3, 2025 @ 23:29:59.287",MAS-MOTAHAREH-M$,DC2K19-01,192.168.173.20,authentication,logged-in,Network,0x0,Kerberos,Logon
1,"Nov 3, 2025 @ 23:29:58.177",DC2K22$,DC2K19-01,-,authentication,logged-out,Network,0x30aebb5fe,-,Logoff
2,"Nov 3, 2025 @ 23:29:56.838",DC2K19-01$,DC2K22,-,authentication,logged-out,Network,0xc68d2ab3,-,Logoff
3,"Nov 3, 2025 @ 23:29:54.173",DC2K19-01$,DC2K22,192.168.0.41,authentication,logged-in,Network,0x0,Kerberos,Logon
4,"Nov 3, 2025 @ 23:29:54.167",DC2K22$,DC2K19-01,192.168.0.40,authentication,logged-in,Network,0x0,Kerberos,Logon



👀 پنج ردیف آخر:


,@timestamp,user.name,host.hostname,source.ip,event.category,event.action,winlog.logon.type,winlog.logon.id,winlog.event_data.LogonProcessName,winlog.task
3742434,"Nov 1, 2025 @ 08:00:00.764",HN-J-JAVDAN$,DC2K19-01,-,authentication,logged-out,Network,0x2fdd02fb9,-,Logoff
3742435,"Nov 1, 2025 @ 08:00:00.751",HN-J-JAVDAN$,DC2K19-01,192.168.189.31,authentication,logged-in,Network,0x0,Kerberos,Logon
3742436,"Nov 1, 2025 @ 08:00:00.711",DC2K22$,DC2K22,192.168.0.40,authentication,logged-in,Network,0x0,Kerberos,Logon
3742437,"Nov 1, 2025 @ 08:00:00.399",ATSH-N-AMERI$,DC2K19-01,192.168.182.71,authentication,logged-in,Network,0x0,Kerberos,Logon
3742438,"Nov 1, 2025 @ 08:00:00.051",FZS-H-REZAPOUR$,DC2K19-01,-,authentication,logged-out,Network,0x2fdd02494,-,Logoff


In [5]:
# 📦 Cell 2 — Preprocess timestamps & basic UBA flags

import pandas as pd
import numpy as np

print("✅ شروع پردازش اولیه دیتاست...")

# یک کپی خام نگه می‌داریم اگر بعداً لازم شد برگردیم
df_raw = df.copy()

# 1) تبدیل @timestamp به datetime
# فرمت فعلی شبیه: "Nov 3, 2025 @ 23:29:59.287"
# اول "@ " را حذف می‌کنیم تا pandas راحت‌تر پارس کند
ts_clean = df['@timestamp'].str.replace(" @ ", " ", regex=False)

df['ts'] = pd.to_datetime(ts_clean, errors='coerce')

# اگر جایی نتوانست پارس کند، تعدادشان را چاپ می‌کنیم
bad_ts = df['ts'].isna().sum()
print(f"📅 تبدیل timestamp انجام شد. ردیف‌های با تاریخ نامعتبر: {bad_ts}")

# 2) سورت بر اساس زمان
df = df.sort_values('ts').reset_index(drop=True)

# 3) ساخت فیچرهای زمانی
df['date'] = df['ts'].dt.date
df['hour_of_day'] = df['ts'].dt.hour + df['ts'].dt.minute / 60.0
df['day_of_week'] = df['ts'].dt.day_name()
df['day_of_week_num'] = df['ts'].dt.dayofweek  # Monday=0, Sunday=6
df['is_weekend'] = df['day_of_week_num'] >= 5

# 4) ساخت فلگ‌های لاگین / لاگ‌اوت
ea = df['event.action'].fillna("").str.lower()
wt = df['winlog.task'].fillna("").str.lower()

df['is_login'] = (ea.eq('logged-in') | wt.eq('logon'))
df['is_logout'] = (ea.eq('logged-out') | wt.eq('logoff'))

print("\n🔎 خلاصه‌ای از فلگ‌ها:")
print("تعداد login ها:", df['is_login'].sum())
print("تعداد logout ها:", df['is_logout'].sum())

print("\n👀 نمونه ۱۰ ردیف بعد از پردازش:")
display(df[['@timestamp', 'ts', 'user.name', 'host.hostname', 'source.ip',
            'event.action', 'winlog.task', 'winlog.logon.type',
            'hour_of_day', 'day_of_week', 'is_login', 'is_logout']].head(10))


✅ شروع پردازش اولیه دیتاست...
📅 تبدیل timestamp انجام شد. ردیف‌های با تاریخ نامعتبر: 0

🔎 خلاصه‌ای از فلگ‌ها:
تعداد login ها: 1868724
تعداد logout ها: 1873628

👀 نمونه ۱۰ ردیف بعد از پردازش:


,@timestamp,ts,user.name,host.hostname,source.ip,event.action,winlog.task,winlog.logon.type,hour_of_day,day_of_week,is_login,is_logout
0,"Oct 1, 2025 @ 00:00:00.964",2025-10-01 00:00:00.964,a.hashemipour,DC2K22,192.168.60.75,logged-in,Logon,Network,0.0,Wednesday,True,False
1,"Oct 1, 2025 @ 00:00:00.985",2025-10-01 00:00:00.985,a.hashemipour,DC2K22,192.168.60.75,logged-in,Logon,Network,0.0,Wednesday,True,False
2,"Oct 1, 2025 @ 00:00:00.997",2025-10-01 00:00:00.997,a.hashemipour,DC2K22,192.168.60.75,logged-in,Logon,Network,0.0,Wednesday,True,False
3,"Oct 1, 2025 @ 00:00:01.001",2025-10-01 00:00:01.001,a.hashemipour,DC2K22,-,logged-out,Logoff,Network,0.0,Wednesday,False,True
4,"Oct 1, 2025 @ 00:00:01.002",2025-10-01 00:00:01.002,a.hashemipour,DC2K22,-,logged-out,Logoff,Network,0.0,Wednesday,False,True
5,"Oct 1, 2025 @ 00:00:01.775",2025-10-01 00:00:01.775,s.sadeghi,DC2K19-01,10.10.21.4,logged-in,Logon,Network,0.0,Wednesday,True,False
6,"Oct 1, 2025 @ 00:00:01.892",2025-10-01 00:00:01.892,KASPERNEW$,DC2K19-01,192.168.0.122,logged-in,Logon,Network,0.0,Wednesday,True,False
7,"Oct 1, 2025 @ 00:00:01.895",2025-10-01 00:00:01.895,KASPERNEW$,DC2K19-01,-,logged-out,Logoff,Network,0.0,Wednesday,False,True
8,"Oct 1, 2025 @ 00:00:01.901",2025-10-01 00:00:01.901,KASPERNEW$,DC2K19-01,192.168.0.122,logged-in,Logon,Network,0.0,Wednesday,True,False
9,"Oct 1, 2025 @ 00:00:02.001",2025-10-01 00:00:02.001,KASPERNEW$,DC2K19-01,-,logged-out,Logoff,Network,0.0,Wednesday,False,True


In [6]:
# 📦 Cell 3 — Feature Engineering for UBA Isolation Forest

import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("🔧 ساخت فیچرهای خام برای مدل ML...")

df_ml = df.copy()

# فقط لاگین‌ها را نگه می‌داریم چون fingerprint روی login ساخته می‌شود
df_ml = df_ml[df_ml['is_login']].copy()

# ---------------------------
# 1) Encode host.hostname
# ---------------------------
host_enc = LabelEncoder()
df_ml['host_enc'] = host_enc.fit_transform(df_ml['host.hostname'].astype(str))

# ---------------------------
# 2) Encode source.ip
# ---------------------------
ip_enc = LabelEncoder()
df_ml['srcip_enc'] = ip_enc.fit_transform(df_ml['source.ip'].astype(str))

# ---------------------------
# 3) Encode logon type
# ---------------------------
type_enc = LabelEncoder()
df_ml['logon_type_enc'] = type_enc.fit_transform(df_ml['winlog.logon.type'].astype(str))

# ---------------------------
# 4) hour_of_day و day_of_week_num آماده است (از سلول قبل)
# ---------------------------

# ---------------------------
# 5) مرتب سازی برای ایمنی
# ---------------------------
df_ml = df_ml.sort_values('ts').reset_index(drop=True)

print("🎯 فیچرهای نهایی برای مدل:")
display(df_ml[['user.name', 'ts', 'host.hostname', 'source.ip',
               'hour_of_day', 'day_of_week_num',
               'host_enc', 'srcip_enc', 'logon_type_enc']].head(10))

print("🔢 تعداد رکوردهای آماده:", len(df_ml))


🔧 ساخت فیچرهای خام برای مدل ML...
🎯 فیچرهای نهایی برای مدل:


,user.name,ts,host.hostname,source.ip,hour_of_day,day_of_week_num,host_enc,srcip_enc,logon_type_enc
0,a.hashemipour,2025-10-01 00:00:00.964,DC2K22,192.168.60.75,0.0,2,2,1160,1
1,a.hashemipour,2025-10-01 00:00:00.985,DC2K22,192.168.60.75,0.0,2,2,1160,1
2,a.hashemipour,2025-10-01 00:00:00.997,DC2K22,192.168.60.75,0.0,2,2,1160,1
3,s.sadeghi,2025-10-01 00:00:01.775,DC2K19-01,10.10.21.4,0.0,2,1,18,1
4,KASPERNEW$,2025-10-01 00:00:01.892,DC2K19-01,192.168.0.122,0.0,2,1,36,1
5,KASPERNEW$,2025-10-01 00:00:01.901,DC2K19-01,192.168.0.122,0.0,2,1,36,1
6,KASPERNEW$,2025-10-01 00:00:02.004,DC2K19-01,192.168.0.122,0.0,2,1,36,1
7,MALI-2019$,2025-10-01 00:00:02.075,DC2K19-01,10.10.21.4,0.0,2,1,18,1
8,s.sadeghi,2025-10-01 00:00:02.097,DC2K22,10.10.21.4,0.0,2,2,18,1
9,NAS-SHAHRDARI$,2025-10-01 00:00:02.353,DC2K22,192.168.4.4,0.0,2,2,1004,1


🔢 تعداد رکوردهای آماده: 1868724


In [7]:
# 📦 Cell 4 — Train IsolationForest per user and compute anomaly score

from sklearn.ensemble import IsolationForest
import numpy as np
from tqdm import tqdm

print("🤖 شروع آموزش مدل‌های UBA برای هر کاربر...")

df_model = df_ml.copy()

# لیست کاربران
users = df_model['user.name'].unique()

# خروجی نهایی
df_results = []

# فیچرهایی که مدل ML استفاده می‌کند
feature_cols = ['hour_of_day', 'day_of_week_num', 'host_enc', 'srcip_enc', 'logon_type_enc']

for user in tqdm(users):
    user_df = df_model[df_model['user.name'] == user].copy()

    # رکوردهای خیلی کم → مدل خوب نمی‌شود
    if len(user_df) < 20:
        # همه رفتارها را Normal در نظر می‌گیریم
        user_df['anomaly_score'] = 0.0
        df_results.append(user_df)
        continue

    # دیتای train روی همین یوزر
    X = user_df[feature_cols]

    # مدل
    model = IsolationForest(
        n_estimators=100,
        contamination='auto',
        random_state=42
    )

    model.fit(X)

    # anomaly score (هر چه منفی‌تر → مشکوک‌تر)
    scores = model.decision_function(X)

    user_df['anomaly_score'] = scores

    df_results.append(user_df)

# ادغام نتایج همه کاربران
df_uba = pd.concat(df_results, ignore_index=True)

print("✅ مدل‌ها ساخته شد.")
print("📌 شکل دیتافریم UBA:", df_uba.shape)

print("\n👀 نمونه ۱۵ ردیف با anomaly_score:")
display(df_uba[['user.name', 'ts', 'host.hostname', 'source.ip',
                'hour_of_day', 'day_of_week_num',
                'anomaly_score']].head(15))


🤖 شروع آموزش مدل‌های UBA برای هر کاربر...


100%|██████████| 2320/2320 [14:32<00:00,  2.66it/s]


✅ مدل‌ها ساخته شد.
📌 شکل دیتافریم UBA: (1868724, 22)

👀 نمونه ۱۵ ردیف با anomaly_score:


,user.name,ts,host.hostname,source.ip,hour_of_day,day_of_week_num,anomaly_score
0,a.hashemipour,2025-10-01 00:00:00.964,DC2K22,192.168.60.75,0.000000,2,-0.046533
1,a.hashemipour,2025-10-01 00:00:00.985,DC2K22,192.168.60.75,0.000000,2,-0.046533
2,a.hashemipour,2025-10-01 00:00:00.997,DC2K22,192.168.60.75,0.000000,2,-0.046533
3,a.hashemipour,2025-10-01 00:00:03.407,DC2K22,192.168.60.75,0.000000,2,-0.046533
4,a.hashemipour,2025-10-01 00:00:03.416,DC2K22,192.168.60.75,0.000000,2,-0.046533
5,a.hashemipour,2025-10-01 00:02:35.351,DC2K22,192.168.60.75,0.033333,2,-0.044045
6,a.hashemipour,2025-10-01 00:02:35.482,DC2K22,192.168.60.75,0.033333,2,-0.044045
7,a.hashemipour,2025-10-01 00:02:35.494,DC2K22,192.168.60.75,0.033333,2,-0.044045
8,a.hashemipour,2025-10-01 00:02:37.016,DC2K22,192.168.60.75,0.033333,2,-0.044045
9,a.hashemipour,2025-10-01 00:02:37.034,DC2K22,192.168.60.75,0.033333,2,-0.044045


In [8]:
# 📦 Cell 5 — Anomaly flagging with per-user weekend fingerprint + export only anomalies

from google.colab import files
import numpy as np

print("🚨 ساخت ستون is_anomaly با threshold = -0.2 ...")

threshold = -0.2

# 1) شروع از خروجی مدل (df_uba از سلول 4)
df_out = df_uba.copy()

# 2) Fingerprint weekend برای هر کاربر (روی همین df_out)
# weekend_ratio = درصد لاگین‌های آخر هفته برای هر user
weekend_fp = (
    df_out
    .groupby('user.name')['day_of_week_num']
    .apply(lambda x: (x >= 5).mean())  # 5,6 => جمعه/شنبه (یا شنبه/یکشنبه بسته به locale)
    .reset_index(name='weekend_ratio')
)

print("📊 نمونه weekend_ratio:")
display(weekend_fp.head(10))

# Merge fingerprint به df_out
df_out = df_out.merge(weekend_fp, on='user.name', how='left')

# اگر برای کاربری به هر دلیل weekend_ratio نیومد، ۰ در نظر می‌گیریم (یعنی هیچوقت آخر هفته نیومده)
df_out['weekend_ratio'] = df_out['weekend_ratio'].fillna(0.0)

# 3) is_anomaly بر اساس threshold
df_out['is_anomaly'] = df_out['anomaly_score'] < threshold

print("🔎 تعداد رفتارهای مشکوک (is_anomaly = True):", df_out['is_anomaly'].sum())

# 4) ساخت why_flagged با منطق بهبودیافته
def build_reason(row):
    reasons = []

    # ⏰ ساعت غیرعادی نسبت به ساعت اداری (قبل ۶ صبح یا بعد ۸ شب)
    if row['hour_of_day'] < 6 or row['hour_of_day'] > 20:
        reasons.append("unusual_hour")

    # 📅 weekend فقط اگر برای این کاربر نادر باشد
    # weekend_ratio < 0.02 یعنی کمتر از ۲٪ لاگین‌ها در آخر هفته بوده
    if (row['day_of_week_num'] in [5, 6]) and (row['weekend_ratio'] < 0.02):
        reasons.append("unusual_weekend_activity")

    # 🧠 چون Isolation Forest همزمان host/ip/logon_type را در نظر می‌گیرد،
    # اگر score زیر threshold باشد، یعنی این ترکیب host/IP/logon type برای این کاربر نادر بوده
    if row['anomaly_score'] < threshold:
        reasons.append("rare_host_or_ip_or_logon_type")

    # اگر هنوز هیچ توضیحی نداریم ولی انومالی است، یک دلیل کلی می‌گذاریم
    if row['is_anomaly'] and len(reasons) == 0:
        reasons.append("behavior_outside_user_fingerprint")

    return ", ".join(reasons) if reasons else ""

df_out['why_flagged'] = df_out.apply(build_reason, axis=1)

# 5) فقط انومالی‌ها را نگه می‌داریم
df_anom = df_out[df_out['is_anomaly']].copy()

print("📉 تعداد ردیف‌های خروجی (فقط انومالی‌ها):", len(df_anom))

print("\n📝 نمونه ۱۰ ردیف anomaly:")
display(df_anom[['user.name','ts','host.hostname','source.ip',
                 'day_of_week','weekend_ratio',
                 'hour_of_day','anomaly_score','is_anomaly','why_flagged']].head(10))

# 6) ذخیره خروجی نهایی (فقط انومالی‌ها)
csv_file = "uba_anomaly_results_only_anomalies.csv"
xlsx_file = "uba_anomaly_results_only_anomalies.xlsx"

df_anom.to_csv(csv_file, index=False)
df_anom.to_excel(xlsx_file, index=False)

print("\n💾 فایل‌ها ساخته شدند:")
print(f"- {csv_file}")
print(f"- {xlsx_file}")

files.download(csv_file)
files.download(xlsx_file)


🚨 ساخت ستون is_anomaly با threshold = -0.2 ...
📊 نمونه weekend_ratio:


,user.name,weekend_ratio
0,137-F-MOLOUDI$,0.250182
1,137-F-ROSTAMI$,0.702532
2,137-FA-ROSTAMI$,0.179615
3,137-N-ZAMIRI$,0.185433
4,137-Z-JALALII$,0.379195
5,137APPLICATIONS$,0.165823
6,A-FVR-S-GHASEMZ$,0.618287
7,A.Arjmand,0.489316
8,A.Motevalli,0.416667
9,A.Saadati,0.451292


🔎 تعداد رفتارهای مشکوک (is_anomaly = True): 2562
📉 تعداد ردیف‌های خروجی (فقط انومالی‌ها): 2562

📝 نمونه ۱۰ ردیف anomaly:


,user.name,ts,host.hostname,source.ip,day_of_week,weekend_ratio,hour_of_day,anomaly_score,is_anomaly,why_flagged
159,a.hashemipour,2025-10-01 07:39:29.026,DC2K22,10.10.12.5,Wednesday,0.24604,7.65,-0.201580,True,rare_host_or_ip_or_logon_type
180,a.hashemipour,2025-10-01 08:03:12.510,DC2K22,10.10.12.5,Wednesday,0.24604,8.05,-0.200844,True,rare_host_or_ip_or_logon_type
181,a.hashemipour,2025-10-01 08:03:12.684,DC2K19-01,10.10.12.5,Wednesday,0.24604,8.05,-0.201239,True,rare_host_or_ip_or_logon_type
182,a.hashemipour,2025-10-01 08:03:12.740,DC2K22,10.10.12.5,Wednesday,0.24604,8.05,-0.200844,True,rare_host_or_ip_or_logon_type
183,a.hashemipour,2025-10-01 08:03:12.759,DC2K22,10.10.12.5,Wednesday,0.24604,8.05,-0.200844,True,rare_host_or_ip_or_logon_type
184,a.hashemipour,2025-10-01 08:03:12.820,DC2K22,10.10.12.5,Wednesday,0.24604,8.05,-0.200844,True,rare_host_or_ip_or_logon_type
185,a.hashemipour,2025-10-01 08:03:12.852,DC2K22,10.10.12.5,Wednesday,0.24604,8.05,-0.200844,True,rare_host_or_ip_or_logon_type
186,a.hashemipour,2025-10-01 08:03:12.876,DC2K22,10.10.12.5,Wednesday,0.24604,8.05,-0.200844,True,rare_host_or_ip_or_logon_type
187,a.hashemipour,2025-10-01 08:03:12.987,DC2K22,10.10.12.5,Wednesday,0.24604,8.05,-0.200844,True,rare_host_or_ip_or_logon_type
188,a.hashemipour,2025-10-01 08:03:27.557,DC2K22,10.10.12.5,Wednesday,0.24604,8.05,-0.200844,True,rare_host_or_ip_or_logon_type



💾 فایل‌ها ساخته شدند:
- uba_anomaly_results_only_anomalies.csv
- uba_anomaly_results_only_anomalies.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>